# Preprocessing — Encoding, Pipelines, ColumnTransformer

*Part of the ML Course Reference Series.*

## Q6: What is One-Hot Encoding and when do you need it?

Most ML algorithms require numeric input. A categorical column like `['cat', 'dog', 'bird']` can't be passed as-is — encoding it as 1/2/3 would imply an order that doesn't exist.

One-hot encoding creates one binary column per category:

```
cat  → [1, 0, 0]
dog  → [0, 1, 0]
bird → [0, 0, 1]
```

Use `handle_unknown='ignore'` so unseen categories in test data don't crash the model.

In [10]:
from sklearn.preprocessing import OneHotEncoder

animals = pd.DataFrame({'animal': ['cat', 'dog', 'bird', 'dog', 'cat']})

enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded = enc.fit_transform(animals[['animal']])

result = pd.DataFrame(encoded, columns=enc.get_feature_names_out())
print(result)

   animal_bird  animal_cat  animal_dog
0          0.0         1.0         0.0
1          0.0         0.0         1.0
2          1.0         0.0         0.0
3          0.0         0.0         1.0
4          0.0         1.0         0.0


## Q7: What is a sklearn Pipeline and why should you always use one?

A `Pipeline` chains preprocessing steps and a model into a single object. Benefits:

1. **No data leakage** — the scaler/imputer is fitted only on training data inside `cross_val_score` or `GridSearchCV`.
2. **One `.fit()` / `.predict()` call** — cleaner code.
3. **Deployable** — the whole pipeline can be saved and loaded as one unit.

Without a pipeline, if you scale *before* splitting, test-set statistics leak into training — your CV scores are optimistic.

In [11]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('knn',    KNeighborsClassifier(n_neighbors=5))
])

# cross_val_score fits the scaler fresh on each train fold — no leakage
scores = cross_val_score(pipe, X_cancer, y_cancer, cv=5, scoring='accuracy')
print(f'Pipeline CV accuracy: {scores.mean():.3f} ± {scores.std():.3f}')

Pipeline CV accuracy: 0.965 ± 0.010


## Q8: What is ColumnTransformer?

Real datasets have mixed types: some columns are numeric, some categorical. `ColumnTransformer` lets you apply different preprocessing to different columns, then concatenate the results.

```
numeric cols  → impute(median) → StandardScaler  ─┐
                                                    ├→ combined feature matrix
category cols → impute(mode)   → OneHotEncoder   ─┘
```

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

# Synthetic mixed dataset
df = pd.DataFrame({
    'age':    [25, np.nan, 35, 40, 28],
    'income': [50, 60, np.nan, 80, 55],
    'gender': ['M', 'F', 'M', np.nan, 'F'],
    'city':   ['NY', 'LA', 'NY', 'LA', 'NY']
})
y = np.array([0, 1, 0, 1, 0])

num_cols = ['age', 'income']
cat_cols = ['gender', 'city']

num_pipe = Pipeline([('imp', SimpleImputer(strategy='median')),
                     ('scl', StandardScaler())])
cat_pipe = Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                     ('enc', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

pipe = Pipeline([('prep', preprocessor), ('model', DecisionTreeClassifier())])
pipe.fit(df, y)

transformed = preprocessor.fit_transform(df)
print('Output shape:', transformed.shape, '  (2 numeric + 2+2 one-hot columns)')

Output shape: (5, 6)   (2 numeric + 2+2 one-hot columns)


---
# 3. Classification Algorithms